# 17 — MCP hands-on with the official Python SDK

The official SDK gives you `FastMCP` for servers (decorators over functions, like the tool decorators in notebooks 08–13) and `ClientSession` for clients. Unlike the provider notebooks, the live cell here needs **no API key**: the transport is stdio, so the client just spawns the server as a local subprocess.

Install the optional extra first: `pip install -e '.[mcp]'`.

Concept mapping to notebook 16: `@mcp.tool()` ≙ `ToyMCPServer.add_tool`, `@mcp.resource()` ≙ `add_resource`, `ClientSession.initialize()` ≙ `ToyMCPClient.connect()` — same wire messages, real transport and schema generation.

Primary sources: [python-sdk](https://github.com/modelcontextprotocol/python-sdk) and the [server quickstart](https://modelcontextprotocol.io/quickstart/server).

In [ ]:
import sys
from importlib.util import find_spec
from pathlib import Path
sys.path.insert(0, str(Path.cwd() / 'notebooks'))

MCP_AVAILABLE = find_spec('mcp') is not None
if not MCP_AVAILABLE:
    print("Install the optional extra to run the live cell: pip install -e '.[mcp]'")

server_code = '''
from mcp.server.fastmcp import FastMCP

mcp = FastMCP('weather-lab')

@mcp.tool()
def get_forecast(city: str) -> str:
    """Deterministic forecast for a known city."""
    return {'oslo': 'Snow, -3C', 'lisbon': 'Sunny, 21C'}.get(city.lower(), 'unknown city')

@mcp.resource('lab://cities')
def cities() -> str:
    return 'oslo\\nlisbon'

if __name__ == '__main__':
    mcp.run()  # stdio transport by default
'''
print(server_code)

In [ ]:
# Live: spawn the server above as a subprocess and talk MCP to it over stdio.
if MCP_AVAILABLE:
    import tempfile
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    server_path = Path(tempfile.mkdtemp()) / 'weather_server.py'
    server_path.write_text(server_code)

    async def demo():
        params = StdioServerParameters(command=sys.executable, args=[str(server_path)])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                info = await session.initialize()
                print('connected to:', info.serverInfo.name)
                tools = await session.list_tools()
                print('discovered:', [t.name for t in tools.tools])
                result = await session.call_tool('get_forecast', {'city': 'Oslo'})
                print('tool result:', result.content[0].text)

    await demo()
else:
    print('skipped: mcp package not installed')

**Design translation.** The session above is the toy exchange from notebook 16 with real framing: the SDK generated the tool's JSON schema from the type hints and docstring, and the handshake/discovery/call messages are the same methods you printed there. What MCP does *not* solve remains your architecture: the decision loop (01), workflow spine (02), verification (03), and budgets and human gates (05). Frameworks from notebooks 07–14 can consume MCP servers directly — see each framework's MCP integration docs — so a server you write once is reusable across all of them.